# 1) Import Configuration and Functions:

In [0]:
%run ../common/configuration


In [0]:
%run ../common/functions

# 2) Define Sprint Schema:

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType

sprint_schema = StructType([
    StructField("season", IntegerType(), False),
    StructField("round", IntegerType(), True),
    StructField("race_name", StringType(), True),
    StructField("circuit_id", StringType(), False),
    StructField("number", IntegerType(), True),
    StructField("position", IntegerType(), True),
    StructField("position_text", StringType(), True),
    StructField("points", IntegerType(), True),
    StructField("grid", IntegerType(), True),
    StructField("laps", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("driver_id", StringType(), True),
    StructField("code", StringType(), True),
    StructField("given_name", StringType(), True),
    StructField("family_name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("constructor_id", StringType(), True),
    StructField("constructor_name", StringType(), True),
    StructField("time_millis", IntegerType(), True),
    StructField("time_gap", StringType(), True),
    StructField("fastest_lap_number", IntegerType(), True),
    StructField("fastest_lap_time", StringType(), True),

])

sprint_input_path = f"{processed_folder_path}/sprint/csv/sprint.csv"
sprint_df = spark.read \
    .option("header", True) \
    .schema(sprint_schema) \
    .csv(sprint_input_path)


# 3) Transform Sprint Data:

The steps included:

- Drop column "code", "given_name", "family_name", "constructor_name", "nationality", "constructor_name".
- Create Surrogate Key.
- Add Data Source and File Date.
- Fill Null cells with "None".

In [0]:
from pyspark.sql.functions import lit

sprint_with_audit_df = sprint_df \
    .withColumn("data_source", lit(v_data_source)) \
    .withColumn("file_date", lit(v_file_date))

sprint_date_df = add_ingestion_date(sprint_with_audit_df)
sprint_fill_df = sprint_date_df.fillna("None")
sprint_dropped_df = sprint_fill_df.drop("code", "given_name", "family_name", "constructor_name", "nationality", "constructor_name")


sprint_final_df = add_surrogate_key(
    sprint_dropped_df,
    key_column_name="results_sk",
    hash_columns=["season", "round", "race_name", "circuit_id", "number", "position",            "position_text", "points", "grid", "laps", "status", "driver_id", "constructor_id", "time_millis", "time_gap", "fastest_lap_number", "fastest_lap_time"],
)

print("Final columns going into the write:", sprint_final_df.columns)

# 4) Save the Processed Dataset to Delta Lake:

In [0]:
sprint_output_path = f"{processed_folder_path}/sprint/delta"

spark.sql("CREATE DATABASE IF NOT EXISTS f1_processed")

upsert_if_changed_for_seasons(
    input_df=sprint_final_df,
    db_name="f1_processed",
    table_name="sprint",
    output_path=sprint_output_path,
    merge_key_columns=["season", "round"],
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(sprint_output_path))

In [0]:
build_presentation_fact(
    processed_location=f"{processed_folder_path}/sprint/delta",
    presentation_directory=f"{presentation_folder_path}/fact_sprint/delta",
    db_name="f1_presentation",
    table_name="fact_sprint",
    partition_columns=["season"],
)

In [0]:
display(spark.read.format("delta").load(f"{presentation_folder_path}/fact_sprint/delta"))

# 5) Save backup Sprint in CSV format:

In [0]:
import io
import csv

sprint_backup_path = f"{presentation_folder_path}/fact_sprint/csv/fact_sprint.csv"

backup_rows = [row.asDict() for row in sprint_final_df.collect()]
backup_fieldnames = sprint_final_df.columns

backup_buffer = io.StringIO()
backup_writer = csv.DictWriter(backup_buffer, fieldnames=backup_fieldnames)
backup_writer.writeheader()
backup_writer.writerows(backup_rows)

dbutils.fs.put(sprint_backup_path, backup_buffer.getvalue(), overwrite=True)
print(f"backup saved: {sprint_backup_path}")